# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
# Optional visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")
print(f"Published: {metadata.get('datePublished','N/A')}")
print(f"Authors: {[author['@id'] for author in metadata.get('author',[])]}")
print(f"Record sets available: {len(metadata.get('recordSet',[]))} record sets.")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Get record set IDs from metadata
record_set_ids = []
for rs in metadata.get('recordSet', []):
    if isinstance(rs, dict) and '@id' in rs:
        record_set_ids.append(rs['@id'])
    elif isinstance(rs, str):
        record_set_ids.append(rs)

if not record_set_ids:
    # Try alternative access (Croissant 1.0 may include recordSets as subobjects)
    # Explore schema structure
    print("No recordSet IDs found in metadata. Checking dataset.record_sets ...")
    try:
        record_sets_dict = dataset.record_sets_to_fields().keys()
        record_set_ids = list(record_sets_dict)
    except Exception as e:
        print("Error extracting recordSet IDs:", e)
# Output available recordSet IDs
print("Available record set @ids:")
for i, rsid in enumerate(record_set_ids):
    print(f"  [{i}] {rsid}")

# Show example of record structure for each recordSet
for rsid in record_set_ids:
    print(f"\nRecords from record set '{rsid}':")
    try:
        for i, x in enumerate(dataset.records(record_set=rsid)):
            print(f"  Record {i}: {x}")
            if i >= 2:
                break
    except Exception as e:
        print(f"Error loading records for {rsid}: {e}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set in record_set_ids:
    print(f"Loading {record_set} ...")
    try:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"  Columns (@id): {df.columns.tolist()}")
        print(f"  Head:")
        print(df.head())
    except Exception as e:
        print(f"  Failed: {e}")

# Pick first record set for illustration
primary_rs_id = record_set_ids[0] if record_set_ids else None
if primary_rs_id is not None and primary_rs_id in dataframes:
    print(f"\nFields in primary record set '{primary_rs_id}':")
    print(dataframes[primary_rs_id].columns.tolist())
    display(dataframes[primary_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Select numeric and grouping fields using their `@id` from the DataFrame columns.

In [ ]:
# Identify numeric and grouping fields by @id
if primary_rs_id is not None:
    df = dataframes[primary_rs_id]
    print("Field types detection (first row):")
    sample = df.iloc[0] if len(df) else {}
    field_types = {}
    for field in df.columns:
        val = sample.get(field, None)
        # Try to infer numeric
        try:
            float(val)
            field_types[field] = 'numeric'
        except:
            field_types[field] = 'categorical'
    print(field_types)

    # Select a numeric field, e.g., GAD-7 total score
    numeric_fields = [k for k,v in field_types.items() if v=='numeric']
    group_fields = [k for k,v in field_types.items() if v=='categorical']

    numeric_field = numeric_fields[0] if numeric_fields else None
    group_field = None
    # Pick an example group field (e.g., gender or education)
    for gf in group_fields:
        if 'gender' in gf.lower():
            group_field = gf
            break
    if group_field is None and group_fields:
        group_field = group_fields[0]

    # EDA for numeric field
    if numeric_field:
        print(f"Numeric field chosen (@id): {numeric_field}")
        threshold = df[numeric_field].astype(float).mean()
        filtered_df = df[df[numeric_field].astype(float) > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
        ) / filtered_df[numeric_field].astype(float).std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        if group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped average {numeric_field} by {group_field}:")
            print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
if primary_rs_id and numeric_field:
    df = dataframes[primary_rs_id]
    # Histogram of numeric field
    plt.figure(figsize=(8,5))
    df[numeric_field] = df[numeric_field].astype(float)
    sns.histplot(df[numeric_field], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field} (by @id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group field if possible
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field} (both by @id)")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.
- Loaded metadata and records using the Croissant schema and `mlcroissant`.
- Record sets and their fields referenced by `@id`, following dataset structure.
- Performed basic filtering, normalization, grouping, and visualization of survey scores.
- The dataset enables detailed exploration of mental health survey responses for Kilifi County, Kenya, with demographic grouping and AI-ready structure.